# SKAT/SKAT-O with LOF variants

- Date Updated: 7/27/2026
- Description: Run Skat and SKAT-O tests with LoF variants and MAX-MAF 0.1% Same script was run for All of Us

In [1]:
# System
import os
import sys

# Dataframe and array manipulation
import pandas as pd
import numpy as np
import math

# Path management
import pathlib
from pathlib import Path
import glob

# Statistics
import statsmodels.api as sm
import scipy
from scipy import stats

# Process management
import shutil
import subprocess

# Workflow management
import papermill as pm

# Visualization
#import matplotlib
#import matplotlib.pyplot as plt
#import seaborn as sns

# Date and time
from datetime import datetime

# Get the timestamp
d = datetime.now().strftime("%m%d%Y")
t = datetime.now().strftime("%m%d%Y_%H%M%S")
print("Last iteration date:", d, "EST")
print("Last iteration time:", t, "EST")

# Get package versions
packages = [pd, np, sm, scipy, pm]

print("\nPackage versions:\n")
for p in packages:
    print("-", p.__name__, p.__version__)

Last iteration date: 06092026 EST
Last iteration time: 06092026_133921 EST

Package versions:

- pandas 2.2.1
- numpy 1.26.0
- statsmodels.api 0.14.4
- scipy 1.15.2
- papermill 2.7.0


In [2]:
BASE_DIR = f"/home/jupyter/ITSN1_Project_Rerun/"

ancestry_list = ["AAC", "AFR", "AJ", "AMR", "CAS", "EAS", "EUR", "MDE", "SAS"]
GENE = "ITSN1"
HOME = "/home/jupyter"
DATA_DIR = f"{HOME}/workspace/"
RELEASE = str(11)
STARTBP = "33642501"
ENDBP = "33899861"
CHROM= str(21)
DT = "wgs"
TOOLS_DIR = f"{HOME}/tools/"
for ancestry in ancestry_list:
    os.makedirs(f"{BASE_DIR}/{ancestry}", exist_ok=True)

In [7]:
# copy over pfiles previously generated in the LD matrices directory, print out 
META_DIR = "/home/jupyter/LD_Matrices_ITSN1"
for ancestry in ancestry_list:
    subprocess.run(["cp", f"{META_DIR}/{ancestry}/chr21_{ancestry}.pgen",
                    f"{META_DIR}/{ancestry}/chr21_{ancestry}.psam",
                    f"{META_DIR}/{ancestry}/chr21_{ancestry}.pvar",
                    f"{META_DIR}/{ancestry}/{ancestry}_pheno.txt",
                    f"{META_DIR}/{ancestry}/{ancestry}_covariate_file.txt",
                    f"{BASE_DIR}/{ancestry}/"])

In [11]:
# read in LoF variants found in GP2 WGS and write variant ids
lof = pd.read_csv(f"{BASE_DIR}/GP2_WGS_LoF_variants_vep_for_remeta.csv", usecols=["#Uploaded_variation"])
lof_variants = lof["#Uploaded_variation"].str.lstrip("chr")
# write lof_variants to file
lof_variants.to_csv(f"{BASE_DIR}/GP2_lof_variants_to_extract.txt", sep="\t")

In [16]:
# create vcf files for rvtests, extracting LoF variants
for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    try:
        subprocess.run(
            [
                "/home/jupyter/tools/plink2",
                "--pfile", f"{WORK_DIR}/chr21_{ancestry}",
                "--pheno", f"{WORK_DIR}/{ancestry}_pheno.txt",
                "--extract", f"{BASE_DIR}/GP2_lof_variants_to_extract.txt",
                "--pheno-name", "PHENO",
                "--mac", "1",
                "--max-maf", "0.001",
                "--make-pgen",
                "--out", f"{WORK_DIR}/{ancestry}_max_maf_lof",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"{ancestry} has no LOF variants. Remove from analysis.")
        print(e.stderr)

AAC has no LOF variants. Remove from analysis.
Error: No variants remaining after main filters.

AMR has no LOF variants. Remove from analysis.
Error: No variants remaining after main filters.

SAS has no LOF variants. Remove from analysis.
Error: No variants remaining after main filters.



In [18]:
# ancestry list
ancestry_list = ["AFR", "AJ", "CAS", "EAS", "EUR", "MDE"]

In [26]:
# convert files to VCF
for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    subprocess.run(
        ["/home/jupyter/tools/plink2",
         "--pfile", f"{WORK_DIR}/{ancestry}_max_maf_lof",
         "--recode", "vcf-iid",
         "--out", f"{WORK_DIR}/{ancestry}_{GENE}.lof"
        ],
        check=True,
        capture_output=True,
        text=True)

In [28]:
# tabix and bgzip files
for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    ! bgzip -f {WORK_DIR}/{ancestry}_{GENE}.lof.vcf
    ! tabix -f -p vcf {WORK_DIR}/{ancestry}_{GENE}.lof.vcf.gz

In [ ]:
# get hg38 refFlat file from ucsc
! wget -nc -O /home/jupyter/refFlat.txt.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz
! gunzip -q -f -k /home/jupyter/refFlat.txt.gz

## RV Tests on ancestries passing filter

In [11]:
for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    # because of errors with rvtests reading in phenotypes, create cov file that matches rvtest pheno file header
    rv_df = pd.read_csv(f'{WORK_DIR}/{ancestry}_covariate_file.txt', sep="\t")
    rv_df.columns = rv_df.columns.str.lower()
    rv_df["fid"] = rv_df["#fid"]
    rv_df["fatid"] = 0
    rv_df["matid"] = 0
    rv_df = rv_df[["fid", "iid", "fatid", "matid", "sex", "pheno", "age", "pc1", "pc2", "pc3", "pc4", "pc5"]]
    rv_df.to_csv(f'{WORK_DIR}/{ancestry}_rvtests_covariate.txt', sep='\t', na_rep="NA", index=False, header=True)

In [ ]:
## RVtests with covariates 
for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    ! /home/jupyter/tools/rvtests/executable/rvtest --noweb --hide-covar \
    --out {WORK_DIR}/{ancestry}_{GENE}.burden.lof \
    --kernel skat,skato \
    --inVcf {WORK_DIR}/{ancestry}_{GENE}.lof.vcf.gz \
    --pheno {WORK_DIR}/{ancestry}_rvtests_covariate.txt \
    --pheno-name pheno \
    --gene {GENE} \
    --geneFile /home/jupyter/refFlat.txt \
    --covar {WORK_DIR}/{ancestry}_rvtests_covariate.txt \
    --covar-name sex,pc1,pc2,pc3,pc4,pc5
    
    # --out : Name of output 
    # --burden cmc --kernel skato: tests to run 
    # --inVcf : VCF file 
    # --gene: gene name (if only looking at one or a few)
    # --geneFile refFlat.txt
    # --pheno :  covar file
    # --mpheno : # column that has phenotype information
    # --pheno-name : column name with phenotype in file
    # --covar : covar file
    # --freqUpper : optional, MAF cut-off
    # --covar-name : covariates, listed by column name, separated by commas (no spaces between commas)
    ## 1=controls; 2=cases

In [ ]:
# 
for ancestry in ancestry_list:
    ! cat {BASE_DIR}/{ancestry}/{ancestry}_{GENE}.burden.lof.Skat.assoc > 
    ! cat {BASE_DIR}/{ancestry}/{ancestry}_{GENE}.burden.lof.SkatO.assoc > 

In [12]:
# read in missense variants found in GP2 WGS and write variant ids
missense = pd.read_csv(f"{BASE_DIR}/GP2_WGS_missense_variants_vep_for_remeta.csv", usecols=["#Uploaded_variation"])
missense_variants = missense["#Uploaded_variation"].str.lstrip("chr")
# write missense_variants to file
missense_variants.to_csv(f"{BASE_DIR}/GP2_missense_variants_to_extract.txt", sep="\t", index=False)

# create vcf files for rvtests, extracting missense variants

for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    try:
        subprocess.run(
            [
                "/home/jupyter/tools/plink2",
                "--pfile", f"{WORK_DIR}/chr21_{ancestry}",
                "--pheno", f"{WORK_DIR}/{ancestry}_pheno.txt",
                "--extract", f"{BASE_DIR}/GP2_missense_variants_to_extract.txt",
                "--pheno-name", "PHENO",
                "--mac", "1",
                "--make-pgen",
                "--out", f"{WORK_DIR}/{ancestry}_missense",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"{ancestry} has no missense variants. Remove from analysis.")
        print(e.stderr)
        
# convert files to VCF
for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    subprocess.run(
        ["/home/jupyter/tools/plink2",
         "--pfile", f"{WORK_DIR}/{ancestry}_missense",
         "--recode", "vcf-iid",
         "--out", f"{WORK_DIR}/{ancestry}_{GENE}.missense"
        ],
        check=True,
        capture_output=True,
        text=True)

# tabix and bgzip files
for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    ! bgzip -f {WORK_DIR}/{ancestry}_{GENE}.missense.vcf
    ! tabix -f -p vcf {WORK_DIR}/{ancestry}_{GENE}.missense.vcf.gz


In [ ]:
## RVtests with covariates 
for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    ! /home/jupyter/tools/rvtests/executable/rvtest --noweb --hide-covar \
    --out {WORK_DIR}/{ancestry}_{GENE}.burden.missense \
    --kernel skat,skato \
    --inVcf {WORK_DIR}/{ancestry}_{GENE}.missense.vcf.gz \
    --pheno {WORK_DIR}/{ancestry}_rvtests_covariate.txt \
    --pheno-name pheno \
    --gene {GENE} \
    --geneFile /home/jupyter/refFlat.txt \
    --covar {WORK_DIR}/{ancestry}_rvtests_covariate.txt \
    --covar-name sex,pc1,pc2,pc3,pc4,pc5
    
    # --out : Name of output 
    # --burden cmc --kernel skato: tests to run 
    # --inVcf : VCF file 
    # --gene: gene name (if only looking at one or a few)
    # --geneFile refFlat.txt
    # --pheno :  covar file
    # --mpheno : # column that has phenotype information
    # --pheno-name : column name with phenotype in file
    # --covar : covar file
    # --freqUpper : optional, MAF cut-off
    # --covar-name : covariates, listed by column name, separated by commas (no spaces between commas)
    ## 1=controls; 2=cases

In [15]:
! cat {BASE_DIR}/*/*_{GENE}.burden.missense.Skat.assoc > {BASE_DIR}/merged_missense.Skat.assoc
! cat {BASE_DIR}/*/*_{GENE}.burden.missense.SkatO.assoc > {BASE_DIR}/merged_missense.SkatO.assoc
! cat {BASE_DIR}/*/*_{GENE}.burden.lof.Skat.assoc > {BASE_DIR}/merged_lof.Skat.assoc
! cat {BASE_DIR}/*/*_{GENE}.burden.lof.SkatO.assoc > {BASE_DIR}/merged_lof.SkatO.assoc